In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
            .config("spark.driver.memory", "2g") \
            .appName("demo").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/22 04:54:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
json_path = "../data/sample_ecommerce_data.json"
df2 = spark.read.option("multiline", "true").json(json_path)

In [7]:
df2.printSchema()

root
 |-- customer: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- price: long (nullable = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment_status: string (nullable = true)



In [10]:
df2.head(10)

[Row(customer=Row(city='Bangalore', customer_id='C001', name='Rahul Sharma'), delivery_status='DELIVERED', items=[Row(category='Electronics', price=70000, product_id='P101', qty=1), Row(category='Electronics', price=500, product_id='P102', qty=2)], order_date='2026-08-20', order_id='ORD001', payment_status='SUCCESS'),
 Row(customer=Row(city='Hyderabad', customer_id='C002', name='Priya Verma'), delivery_status='PENDING', items=[Row(category='Fashion', price=3000, product_id='P103', qty=1)], order_date='2026-08-21', order_id='ORD002', payment_status='FAILED'),
 Row(customer=Row(city='Delhi', customer_id='C003', name='Amit Singh'), delivery_status='SHIPPED', items=[Row(category='Grocery', price=200, product_id='P104', qty=5)], order_date='2026-08-21', order_id='ORD003', payment_status='SUCCESS'),
 Row(customer=Row(city='Bangalore', customer_id='C001', name='Rahul Sharma'), delivery_status='DELIVERED', items=[Row(category='Electronics', price=15000, product_id='P105', qty=1)], order_date='

In [25]:
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType, DateType


In [29]:
cust_schema = StructType(
    [StructField("city",StringType(),True),
     StructField("customer_id",StringType(),True),
     StructField("name",StringType(),True)]
)
element_schema = StructType(
    [StructField("category",StringType(),True),
     StructField("price",IntegerType(),True),
     StructField("product_id",StringType(),True),
     StructField("qty",IntegerType(),True)
    ]
)
items_schema = ArrayType(element_schema)

final_schema = StructType(
    [StructField("customer",cust_schema,True),
     StructField("delivery_status",StringType(),True),
     StructField("items",items_schema,True),
     StructField("order_date",DateType(),True),
     StructField("order_id",StringType(),True),
     StructField("payment_status",StringType(),True)
    ]
)


In [30]:
df3 = spark.read\
            .format("json")\
            .option("multiline",True)\
            .schema(final_schema)\
            .load(json_path)

In [31]:
df3.head()

Row(customer=Row(city='Bangalore', customer_id='C001', name='Rahul Sharma'), delivery_status='DELIVERED', items=[Row(category='Electronics', price=70000, product_id='P101', qty=1), Row(category='Electronics', price=500, product_id='P102', qty=2)], order_date=datetime.date(2026, 8, 20), order_id='ORD001', payment_status='SUCCESS')

In [34]:
import pyspark.sql.functions as F

In [44]:
df_exploded = df3.withColumn("item", F.explode("items"))

In [36]:
df_exploded = df_exploded.drop("items")

In [45]:
df_exploded.head(5)

[Row(customer=Row(city='Bangalore', customer_id='C001', name='Rahul Sharma'), delivery_status='DELIVERED', items=[Row(category='Electronics', price=70000, product_id='P101', qty=1), Row(category='Electronics', price=500, product_id='P102', qty=2)], order_date=datetime.date(2026, 8, 20), order_id='ORD001', payment_status='SUCCESS', item=Row(category='Electronics', price=70000, product_id='P101', qty=1)),
 Row(customer=Row(city='Bangalore', customer_id='C001', name='Rahul Sharma'), delivery_status='DELIVERED', items=[Row(category='Electronics', price=70000, product_id='P101', qty=1), Row(category='Electronics', price=500, product_id='P102', qty=2)], order_date=datetime.date(2026, 8, 20), order_id='ORD001', payment_status='SUCCESS', item=Row(category='Electronics', price=500, product_id='P102', qty=2)),
 Row(customer=Row(city='Hyderabad', customer_id='C002', name='Priya Verma'), delivery_status='PENDING', items=[Row(category='Fashion', price=3000, product_id='P103', qty=1)], order_date=da

In [46]:
df_exploded.count()

10

In [48]:
df_exploded.select(F.col("customer.customer_id") , F.col("item.product_id")).show(15)

+-----------+----------+
|customer_id|product_id|
+-----------+----------+
|       C001|      P101|
|       C001|      P102|
|       C002|      P103|
|       C003|      P104|
|       C001|      P105|
|       C004|      P106|
|       C004|      P107|
|       C006|      P108|
|       C002|      P109|
|       C008|      P110|
+-----------+----------+



In [49]:
df3.count()

10